# 00 · Start here: your Kafka practice environment
**Level:** beginner · **Time:** 20–30 minutes · **Prerequisites:** basic Python, a terminal.

You will check your Python kernel, inspect the workshop, and verify a Kafka connection.
The running example throughout the course is an online shop: orders become events, and
independent consumers turn those events into useful views.

Run one cell at a time with **Shift+Enter**. Read its output before continuing. Use
**Kernel → Restart Kernel and Run All Cells** to repeat a lesson. Every lesson creates
fresh topic names; a fresh group reads retained records independently. Do not run
several copies of the same Django notebook against the same database concurrently.


## 1. Install once, from the project folder
Install Python 3.11–3.13 and Docker Desktop with Linux containers. Start Docker Desktop.
The commands below are **terminal commands**, not Python cells.

**Windows PowerShell** (activation is unnecessary):
```powershell
python -m venv .venv
.\.venv\Scripts\python.exe -m pip install -r requirements.txt
Copy-Item .env.example .env
docker compose up -d --wait
.\.venv\Scripts\python.exe -m ipykernel install --sys-prefix --name kafka-workshop --display-name "Python (Kafka Workshop)"
.\.venv\Scripts\python.exe -m jupyter lab
```
**macOS/Linux:** use `.venv/bin/python` in place of `.\.venv\Scripts\python.exe`,
and `cp .env.example .env`. Choose **Python (Kafka Workshop)** in the notebook kernel menu.
The initial image/package downloads require internet access. Python connects to
`localhost:9092`; containers use `kafka:19092` internally.


In [ ]:
from pathlib import Path
import sys
from uuid import uuid4

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "pyproject.toml").exists() and (p / "labtools").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open Jupyter from the kafka-django workshop folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from labtools.kafka import (
    bootstrap_servers, client_config, require_broker, new_topic,
    producer, consumer, publish, poll_records, collect, decode,
)
RUN = uuid4().hex[:8]
print("Workshop:", ROOT.name, "| run:", RUN)


## 2. Confirm the kernel
`sys.executable` should point inside this project's `.venv`. If an import fails, choose
the correct kernel; installing into an unrelated terminal Python will not repair it.


In [ ]:
import platform
from importlib.metadata import version
print("Python:", platform.python_version())
print("Kernel executable:", sys.executable)
for package in ["confluent-kafka", "Django", "jupyterlab", "jsonschema"]:
    print(f"{package}: {version(package)}")
assert sys.version_info >= (3, 11)


## 3. Learn the five nouns
| Term | In this workshop |
|---|---|
| Producer | Python/Django code sending an order event |
| Topic | Named log of related events |
| Partition | Ordered portion of a topic; unit of consumer parallelism |
| Offset | A record's position within **one** partition |
| Consumer group | Collaborating consumers with their own saved progress |

A Kafka read does not remove a record. Retention/compaction decides when data is removed.
Two groups can process the same events independently. Ordering is per partition, not global.


In [ ]:
lesson_files = sorted((ROOT / "notebooks").glob("*.ipynb"))
for path in lesson_files:
    print(path.name)
print("Notebook count:", len(lesson_files))


## 4. Check Docker without starting or stopping anything
Expected: a running, healthy `kafka` service. This diagnostic still works when Docker
is absent: it explains the next step instead of claiming that Kafka is running.


In [ ]:
import shutil
import subprocess
if shutil.which("docker"):
    check = subprocess.run(["docker", "compose", "ps"], cwd=ROOT,
                           capture_output=True, text=True, timeout=20)
    print(check.stdout or check.stderr)
else:
    print("Docker CLI is not available. Install/start Docker Desktop, then reopen your terminal.")


## 5. Ask Kafka for metadata
Expected: broker `1` and its advertised address. This notebook catches the connection
error for diagnosis. All subsequent Kafka labs require a successful connection.


In [ ]:
BROKER_READY = False
try:
    metadata = require_broker(timeout=5)
    BROKER_READY = True
    print("Brokers:", {key: str(value) for key, value in metadata.brokers.items()})
except RuntimeError as error:
    print(error)
print("Ready for notebook 01:", BROKER_READY)


## 6. Choose a learning route
* **First working demo:** 00 → 01 → 02 → 03 → 04 (about 3 hours).
* **Application developer:** add 05 → 06 → 08 → 11 → 12 → 13 → 14.
* **Advanced practice:** complete 07 → 09 → 10 → 15 as well.

Allow roughly 12–16 hours for all guided labs, plus time for exercises. A notebook is
a starting point for practice, not a certification of production expertise. Kafka Connect,
Schema Registry deployment, cross-region replication and full cluster security are extension topics.


## Exercise
Explain why two partitions may both contain offset `0`. Then explain what happens if
two independent consumer groups subscribe to the same topic.

<details><summary>Solution hints</summary>
Offsets are scoped to a topic-partition. Each group has independent committed offsets,
so both groups can read the same retained events. Within a traditional consumer group,
one partition is assigned to at most one active member at a time.
</details>

## Troubleshooting and cleanup
* Connection refused: `docker compose ps`, then `docker compose logs --tail 80 kafka`.
* Kafka connects then fails: check advertised listeners; host clients need `localhost:9092`.
* Import error: check the printed kernel path and reinstall requirements into that interpreter.
* A polling timeout: check the expected count, group offsets, transaction isolation and broker health.
* Stop while keeping records: `docker compose down`.
* Delete **only this workshop's** broker data: `docker compose down -v` (deliberately destructive).

See [the Apache quickstart](https://kafka.apache.org/41/getting-started/quickstart/)
and [the Python client guide](https://docs.confluent.io/kafka-clients/python/current/overview.html).
Next: **01 — first producer and consumer**.
